# Sensitivity to the baseline reference temperature

Rebuild the adaptive baseline for each integer reference temperature from
10 to 20 °C using the same 50 nine-day hybrid checkpoints. Reference
relative humidity stays at 50%, matching the saved configuration. The
comparison uses the same population, surveillance inputs, evaluation dates
and mean-alignment rule for every reference. The heat and cold
counterfactual thresholds stay at 20 °C and 10 °C, respectively.

The table reports daily and weekly RMSE computed after averaging member
predictions, the mean daily population-weighted baseline rate, and
cumulative model-attributed heat and cold deaths. Differences use the
original 16 °C reference as the comparison. Run the full sweep on HPC.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

cwd = Path.cwd()
repo = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "configs" / "training.example.yaml").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from the sari-weather-mortality checkout.")

sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load helpers from this checkout even if Jupyter cached another notebooks package.
utils_path = repo / "notebooks" / "utils.py"
spec = spec_from_file_location("sari_notebook_utils", utils_path)
notebook_utils = module_from_spec(spec)
spec.loader.exec_module(notebook_utils)

ANALYSIS_PERIOD = notebook_utils.ANALYSIS_PERIOD
daily_index = notebook_utils.daily_index
checkpoint_paths = notebook_utils.checkpoint_paths
ensemble = notebook_utils.ensemble
national_daily = notebook_utils.national_daily
weekly_sum = notebook_utils.weekly_sum
DEVICE = notebook_utils.DEVICE

from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor

from dataclasses import replace
from sari_t.reference import ReferenceConfig

## Load the model for inspection

This cell loads the selected checkpoint onto CPU. Inspect the predictor
as `model`. The evaluation below uses all 50 nine-day hybrid members.

In [ ]:
inspection_member = 0
paths = checkpoint_paths("default")
config = load_run_config(paths[0].parents[1] / "config.yaml")
if config.model_config.kernel_T_days != 9 or not config.training_config.baseline_adaptation:
    raise ValueError("Select the nine-day adaptive hybrid ensemble.")
if config.reference_config != ReferenceConfig(temperature_c=16, relative_humidity_percent=50):
    raise ValueError("Expected the released 16 °C / 50% RH hybrid reference.")
model = restore_predictor(load_checkpoint(paths[inspection_member]), device="cpu").eval()
model

## Rebuild the baseline for each reference

In [ ]:
dates = daily_index()
if len(dates) % 7 or dates[0].dayofweek != 0:
    raise ValueError("Expected complete Monday–Sunday analysis weeks.")
scenarios = {
    "heat_20": {"maximum_c": 20},
    "cold_10": {"minimum_c": 10},
}

def summarize(workspace, result):
    daily = national_daily(workspace, result)
    national_population = workspace.data.population.sum(dim=1)
    daily["Baseline rate"] = (
        (workspace.baseline_rate.squeeze(1) * national_population).sum(dim=(1, 2))
        / national_population.sum(dim=(1, 2)) * 100_000
    )
    daily["Observed weekly"] = result.labels.weekly_de_age_sex.sum(dim=(1, 2))
    for name in scenarios:
        daily[name] = daily["Predicted"] - result.scenarios[name].sum(dim=(1, 2, 3))
    return daily

inference_model = InferenceModel.from_checkpoint(
    paths[0], data_config=config.data_config, period=ANALYSIS_PERIOD, device=DEVICE,
)
reconstructions = {}
rows = []
for temperature in range(10, 21):
    reference = replace(config.reference_config, temperature_c=temperature)
    print(f"Evaluating reference {temperature} °C", flush=True)
    summary = ensemble(
        inference_model, paths, summarize, scenarios=scenarios, baseline_reference=reference,
    )
    reconstructions[temperature] = summary
    daily_error = summary["Predicted"] - summary["Observed"]
    weekly_error = weekly_sum(summary["Predicted"]) - summary["Observed weekly"]
    rows.append({
        "Reference (°C)": temperature,
        "Daily RMSE (deaths/day)": daily_error.square().mean().sqrt().item(),
        "Weekly RMSE (deaths/week)": weekly_error.square().mean().sqrt().item(),
        "Baseline rate (per 100,000/day)": summary["Baseline rate"].mean().item(),
        "Heat-associated deaths": summary["heat_20"].sum().item(),
        "Cold-associated deaths": summary["cold_10"].sum().item(),
    })

reference_results = pd.DataFrame(rows).set_index("Reference (°C)")
difference_from_16 = reference_results.subtract(reference_results.loc[16], axis="columns")
display(reference_results.round(2))
display(difference_from_16.round(2).style.set_caption("Difference from the 16 °C reference"))

## Reconstruction, baseline and attribution across reference temperatures

In [ ]:
with plt.rc_context({"font.family": "DejaVu Serif", "axes.axisbelow": True}):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
    panels = [
        (["Daily RMSE (deaths/day)"], "Daily reconstruction", "RMSE (deaths/day)"),
        (["Weekly RMSE (deaths/week)"], "Weekly reconstruction", "RMSE (deaths/week)"),
        (["Baseline rate (per 100,000/day)"], "Adaptive baseline", "Rate (per 100,000/day)"),
        (["Heat-associated deaths", "Cold-associated deaths"],
         "Weather attribution", "Cumulative deaths"),
    ]
    for ax, (columns, title, ylabel) in zip(axes.flat, panels):
        for column in columns:
            color = "#D73027" if column.startswith("Heat") else "#2A5C9E"
            ax.plot(reference_results.index, reference_results[column],
                    color=color, marker="o", linewidth=1.5, label=column)
        ax.axvline(16, color="0.5", linestyle=":", linewidth=1, label="Original reference")
        ax.set(title=title, ylabel=ylabel)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
        if len(columns) > 1:
            ax.legend(frameon=False)

    for ax in axes[-1]:
        ax.set(xlabel="Baseline reference temperature (°C)", xticks=range(10, 21))
    fig.suptitle("Sensitivity to the baseline reference temperature", fontsize=16)
    fig.tight_layout()
    plt.show()